# Multi-Circuit Resource Estimation Benchmarking

This notebook benchmarks resource estimation across **multiple circuits** using
the existing Azure QDK and Qualtran estimators. For every circuit it:

1. Loads (or generates) the circuit.
2. Transpiles to Clifford+T.
3. Runs **one Azure** and **one Qualtran** estimate.
4. Collects all returned metrics into a single comprehensive DataFrame.
5. Generates **three** comparison plots (Azure vs Qualtran):
   - **T Count vs Space-Time**
   - **T Count vs Physical Qubits** (stacked compute + factory area)
   - **T Count vs Runtime**

All estimator logic, transpilation, synthesis, and configuration are left
unchanged — only the notebook's plotting and data selection are refined.


In [1]:

import sys
import pathlib

# Locate the repo root
_search = pathlib.Path.cwd()
for _ in range(8):
    if (_search / "resourceEstimationPipeline").is_dir():
        REPO_ROOT = str(_search)
        break
    _search = _search.parent
else:
    REPO_ROOT = str(pathlib.Path.cwd())

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:.4g}".format)

import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# ── Pipeline core modules ─────────────────────────────────────────────────────
from resourceEstimationPipeline.config import (
    PipelineConfig,
    TranspileConfig,
    AzureConfig,
    QualtranConfig,
)
from resourceEstimationPipeline.circuit.transpile import (
    transpile_to_clifford_t,
    circuit_stats,
    circuit_to_qasm,
)
from resourceEstimationPipeline.compare.metrics import compare, enrich_from_circuit

# ── Bridge helper (inject Azure params -> Qualtran config) ─────────────────────
from resourceEstimationPipeline.estimators.azure import apply_azure_to_qualtran

# ── Estimators (imported lazily inside run_estimation per circuit) ─────────────

print(f"REPO_ROOT: {REPO_ROOT}")
print("Imports OK")


REPO_ROOT: /Users/kevinli/Documents/resourceEstimation
Imports OK


In [2]:

cfg = PipelineConfig(
    transpile=TranspileConfig(
        optimization_level=1,
        seed_transpiler=42,
        rotation_synthesis_enabled=False,
        rotation_synthesis_epsilon=1e-4,
        synthesis_strategy="qiskit_synth",
        synthesis_method='pygridsynth',
    ),
    azure=AzureConfig(
        error_budget=0.01,
        error_rate=1e-3,
        gate_time_ns=50.0,
        measurement_time_ns=100.0,
        factory_type="RoundBased",
        slow_down_factors=[1.0, 1.5, 2.0],
        optimization_level=1,
        use_graph=False,
        minimize="qubit_hours",
        pareto_index=0,
    ),
    qualtran=QualtranConfig(
        data_d=23,
        phys_err=1e-3,
        error_budget=0.01,
        data_block="compact",
        factory_type="15to1",
        n_factories=6,
        use_gidney_fowler=False,
        use_beverland=True,
        use_azure_parameters=True,
        pareto_index=0,
    ),
)

print(cfg)


PipelineConfig(hamlib=HamlibConfig(hdf5_path='./../hamlib/condensedmatter/heisenberg/heis.hdf5', key=None, key_index=313), evolution=EvolutionConfig(evolution_time=1.0, synthesis_order=2, synthesis_reps=10), transpile=TranspileConfig(basis_gates=['cx', 'rz', 'h', 's', 'sdg', 'x', 'y', 'z', 't', 'tdg'], optimization_level=1, seed_transpiler=42, rotation_synthesis_enabled=False, rotation_synthesis_epsilon=0.0001, synthesis_strategy='qiskit_synth', synthesis_method='pygridsynth', pygridsynth_precision=None), azure=AzureConfig(error_budget=0.01, error_rate=0.001, gate_time_ns=50.0, measurement_time_ns=100.0, two_qubit_gate_time_ns=None, code_distance=None, factory_type='RoundBased', slow_down_factors=[1.0, 1.5, 2.0], optimization_level=1, use_graph=False, minimize='qubit_hours', pareto_index=0), qualtran=QualtranConfig(data_d=23, data_d_sweep=None, phys_err=0.001, t_gate_ns=50.0, t_meas_ns=100.0, cycle_time_us=1.0, error_budget=0.01, data_block='compact', factory_type='15to1', qec_scheme='

In [3]:

# ---------------------------------------------------------------------------
# Helper: synthesize synthetic circuits for benchmarking
# ---------------------------------------------------------------------------

def t_circuit_distribution(n_circuits: int, t_max: int, q_max: int, linear: bool = True):
    """Return a list of synthetic Clifford+T circuits.

    Each circuit has *q* qubits and approximately *t* T gates distributed
    linearly or geometrically across the range [1, t_max] x [2, q_max].

    Parameters
    ----------
    n_circuits : int  number of circuits to generate
    t_max      : int  max T gate count per circuit
    q_max      : int  max qubit count per circuit
    linear     : bool  if True use linspace; otherwise use geomspace

    Returns
    -------
    list[QuantumCircuit]
    """
    from qiskit import QuantumCircuit
    import numpy as np

    t_vals = np.linspace(1, t_max, n_circuits) if linear else np.geomspace(1, t_max + 1, n_circuits)
    q_vals = np.linspace(2, q_max, n_circuits) if linear else np.geomspace(2, q_max + 1, n_circuits)

    circuits = []
    for t, q in zip(t_vals, q_vals):
        qc = QuantumCircuit(int(q), 0)
        for i in range(int(t)):
            current = i % int(q)
            nxt = (i + 1) % int(q)
            qc.h(current)
            qc.t(current)
            qc.cx(current, nxt)
        circuits.append(qc)
    return circuits


# ---------------------------------------------------------------------------
# Helper: load a circuit from a .qasm file (optional future extensibility)
# ---------------------------------------------------------------------------

def load_circuit_from_file(path: str):
    """Load a quantum circuit from an OpenQASM file.

    Parameters
    ----------
    path : str  path to a .qasm file

    Returns
    -------
    QuantumCircuit
    """
    from qiskit import QuantumCircuit
    return QuantumCircuit.from_qasm_file(path)


# ---------------------------------------------------------------------------
# Define the benchmark circuits
# ---------------------------------------------------------------------------

c_count = 25
t_count = 1000
q_count = 100

circuit_list = t_circuit_distribution(c_count, t_count, q_count, True)
print(f"Generated {len(circuit_list)} synthetic circuits.")


Generated 25 synthetic circuits.


## Benchmarking Pipeline

The cells below contain **reusable helper functions** extracted from the
one-off workflow. Call `run_benchmark(circuit_list, cfg)` to execute
the full multi-circuit pipeline.


In [4]:

import re

def _safe_get(result, attr, default=None):
    """Safely retrieve an attribute from a possibly-None result."""
    if result is None:
        return default
    return getattr(result, attr, default)


def transpile_circuit(qc, cfg):
    """Transpile a single circuit to Clifford+T.

    Returns (clifford_t_circuit, stats_dict) or (None, {}) on failure.
    """
    try:
        ct = transpile_to_clifford_t(qc, cfg.transpile)
        stats = circuit_stats(ct)
        return ct, stats
    except Exception as exc:
        print(f"  Warning: Transpilation failed: {exc}")
        return None, {}


def run_estimation(ct_circuit, config):
    """Run one Azure and one Qualtran estimate on a single transpiled circuit.

    Returns (azure_result, qualtran_result) - each may be None on failure.
    Both results are enriched with circuit-derived metrics via enrich_from_circuit().
    """
    azure_result = None
    qualtran_result = None

    # -- Azure QDK ---------------------------------------------------------------
    try:
        from resourceEstimationPipeline.estimators.azure import estimate as azure_estimate
        azure_result = azure_estimate(ct_circuit, config)
        azure_result = enrich_from_circuit(azure_result, ct_circuit)
    except ImportError as e:
        print(f"  Warning: Azure unavailable: {e}")
    except Exception as e:
        print(f"  Warning: Azure estimation failed: {e}")

    # -- Bridge: inject Azure params -> Qualtran config --------------------------
    if azure_result is not None and config.qualtran.use_azure_parameters:
        az_d = _safe_get(azure_result, 'code_distance')
        az_n = _safe_get(azure_result, 'num_factories')
        if (az_d is not None or az_n is not None):
            try:
                config = apply_azure_to_qualtran(azure_result, config)
            except Exception as e:
                print(f"  Warning: Azure->Qualtran bridge failed: {e}")

    # -- Qualtran ----------------------------------------------------------------
    try:
        from resourceEstimationPipeline.estimators.qualtran import estimate as qt_estimate
        qualtran_result = qt_estimate(ct_circuit, config)
        qualtran_result = enrich_from_circuit(qualtran_result, ct_circuit)
    except ImportError as e:
        print(f"  Warning: Qualtran unavailable: {e}")
    except Exception as e:
        print(f"  Warning: Qualtran estimation failed: {e}")

    return azure_result, qualtran_result


# ---------------------------------------------------------------------------
# Simplify estimator names for plotting.
# The full name (with config params) is preserved in the DataFrame; this helper
# only extracts the family so that all points from the same estimator produce
# a single continuous line on each plot.
# ---------------------------------------------------------------------------

_FAMILY_RE = re.compile(r"^(Azure|Qualtran)")


def _family(name: str) -> str:
    """Return 'Azure' or 'Qualtran', falling back to the full name."""
    m = _FAMILY_RE.search(name)
    return m.group(1) if m else name


# ---------------------------------------------------------------------------
# collect_metrics: one row per (circuit, estimator) — same format as
# comparison_simple.ipynb but extended with qubit breakdown columns.
# ---------------------------------------------------------------------------

def collect_metrics(circuit_name, azure_result, qualtran_result, ct_stats):
    """Collect metrics into DataFrame rows.

    Keeps the side-by-side format from comparison_simple.ipynb:
      Metric | Azure (d=...,...) | Qualtran (d=...,...) | Ratio
    for every circuit, plus qubit breakdown columns for plotting.

    Missing fields become None -> pandas renders them blank.
    """
    for result in [azure_result, qualtran_result]:
        if result is None:
            continue
        t_count = _safe_get(result, 't_count')
        runtime = _safe_get(result, 'runtime_seconds')
        total_q = _safe_get(result, 'physical_qubits')
        compute_q = _safe_get(result, 'physical_compute_qubits')
        factory_q = _safe_get(result, 'physical_factory_qubits')

        # Space-time volume (qubit-seconds) using the same units each estimator reports.
        if total_q is not None and runtime is not None:
            space_time = float(total_q) * runtime
        else:
            space_time = None

        yield {
            "circuit_name":       circuit_name,
            "estimator_family":   _family(result.estimator_name),
            "t_count":            t_count,
            "clifford_count":     _safe_get(result, 'clifford_count'),
            "rotation_count":     _safe_get(result, 'rotation_count'),
            "toffoli_count":      _safe_get(result, 'toffoli_count'),
            "measurement_count":  _safe_get(result, 'measurement_count'),
            "runtime_seconds":    runtime,
            "total_qubits":       total_q,
            "compute_qubits":     compute_q,
            "factory_qubits":     factory_q,
            "space_time_volume":  space_time,
            "code_distance":      _safe_get(result, 'code_distance'),
            "logical_error_rate": _safe_get(result, 'logical_error_rate'),
            "error_budget":       _safe_get(result, 'error_budget'),
            "physical_error_rate":_safe_get(result, 'physical_error_rate'),
            "logical_qubits":     _safe_get(result, 'logical_qubits'),
            "logical_cycles":     _safe_get(result, 'logical_cycles'),
            "factory_count":      _safe_get(result, 'factory_count'),
            "num_factories":      _safe_get(result, 'num_factories'),
        }


def run_benchmark(circuit_list, config):
    """Run the full multi-circuit benchmarking pipeline.

    Parameters
    ----------
    circuit_list : list[QuantumCircuit] or list[str]  circuits or file paths
    config       : PipelineConfig

    Returns
    -------
    (pandas.DataFrame, dict[str, dict[str, EstimationResult]])
        DataFrame: one row per (circuit, estimator) — same format as
        comparison_simple.ipynb extended with qubit breakdown columns.
        circuit_results: nested dict {circuit_name: {'Azure': result, 'Qualtran': result}}
            so the side-by-side section can use the real EstimationResult objects.

        Columns for plotting:
            circuit_name, estimator_family, t_count, total_qubits,
            compute_qubits, factory_qubits, space_time_volume, runtime_seconds
        Columns from the comparison layer:
            clifford_count, rotation_count, logical_error_rate, code_distance, etc.
    """
    all_rows = []
    circuit_results = {}  # {circuit_name: {'Azure': result, 'Qualtran': result}}

    for idx, item in enumerate(circuit_list):
        # Derive circuit name
        if isinstance(item, str):
            circuit_name = pathlib.Path(item).stem
        elif hasattr(item, 'name') and getattr(item, 'name', None):
            circuit_name = item.name
        else:
            circuit_name = f"circuit_{idx}"

        # Step A: load (if path string) -> transpile
        qc = load_circuit_from_file(item) if isinstance(item, str) else item
        ct_circuit, ct_stats = transpile_circuit(qc, config)
        if ct_circuit is None:
            print(f"  Skipping circuit '{circuit_name}' (transpilation failed).")
            continue

        print(f"[{idx+1}/{len(circuit_list)}] Circuit '{circuit_name}': "
              f"T={ct_stats.get('t_count', '?')}, qubits={ct_stats.get('num_qubits', '?')}")

        # Step B: estimate (one Azure + one Qualtran)
        azure_res, qualtran_res = run_estimation(ct_circuit, config)

        # Save real EstimationResult objects for side-by-side comparison.
        circuit_results[circuit_name] = {}
        if azure_res is not None:
            circuit_results[circuit_name]['Azure'] = azure_res
        if qualtran_res is not None:
            circuit_results[circuit_name]['Qualtran'] = qualtran_res

        # Step C: collect metrics into DataFrame rows
        rows = list(collect_metrics(circuit_name, azure_res, qualtran_res, ct_stats))
        all_rows.extend(rows)

    df = pd.DataFrame(all_rows)
    return df, circuit_results


In [5]:

benchmark_df, circuit_results = run_benchmark(circuit_list, cfg)
print(f"\nBenchmark complete. {len(benchmark_df)} rows collected.")

# -- Display side-by-side comparison for the first circuit -------------------
# Uses the real EstimationResult objects saved during benchmarking — the same
# approach as comparison_simple.ipynb (cells with report, comparison_dataframe, etc.).
from resourceEstimationPipeline.compare.metrics import compare
from resourceEstimationPipeline.compare.tables import (
    comparison_dataframe, differences_dataframe, missing_dataframe, explain_differences,
)

estimator_names_in_df = benchmark_df['estimator_family'].unique()
circuits_list = benchmark_df['circuit_name'].unique()

if len(circuits_list) > 0 and len(estimator_names_in_df) >= 2:
    first_circuit = circuits_list[0]

    # Pull the real EstimationResult objects (not fake wrappers).
    results_map = circuit_results.get(first_circuit, {})
    azure_r = results_map.get('Azure')
    qt_r = results_map.get('Qualtran')

    available = [r for r in [azure_r, qt_r] if r is not None]

    if len(available) >= 2:
        # Build comparison report using the same helper as comparison_simple.ipynb.
        report = compare(available)
        print(f"\nCircuit : {first_circuit}")
        print(f"Estimators compared   : {report.estimator_names}")
        print(f"Shared metrics        : {len(report.shared_metrics)}")
        print(f"N/A in ≥1 estimator   : {len(report.missing_metrics)}")
        print(f"Numeric differences   : {len(report.differences)}")

        # Full comparison table — identical format to comparison_simple.ipynb.
        n_total = len(report.metric_rows)
        display(Markdown(
            f"*{n_total} metrics total — "
            f"**{len(report.shared_metrics)} shared** | "
            f"**{len(report.missing_metrics)} framework-specific*.*"
        ))
        df_comp = comparison_dataframe(report)
        display(df_comp)

        # Metrics that differ between estimators.
        diff_df = differences_dataframe(report)
        if not diff_df.empty:
            print("\nMetrics that differ:")
            display(diff_df)

        # Explanation of differences (same text as comparison_simple.ipynb).
        print(explain_differences(report))
    else:
        print("Need both Azure and Qualtran results for comparison.")
else:
    print("Not enough data for a side-by-side table.")


[1/25] Circuit 'circuit-41': T=1, qubits=2
[2/25] Circuit 'circuit-42': T=42, qubits=6
[3/25] Circuit 'circuit-43': T=84, qubits=10
[4/25] Circuit 'circuit-44': T=125, qubits=14
[5/25] Circuit 'circuit-45': T=167, qubits=18
[6/25] Circuit 'circuit-46': T=209, qubits=22
[7/25] Circuit 'circuit-47': T=250, qubits=26
[8/25] Circuit 'circuit-48': T=292, qubits=30
[9/25] Circuit 'circuit-49': T=334, qubits=34
[10/25] Circuit 'circuit-50': T=375, qubits=38
[11/25] Circuit 'circuit-51': T=417, qubits=42
[12/25] Circuit 'circuit-52': T=458, qubits=46
[13/25] Circuit 'circuit-53': T=500, qubits=51
[14/25] Circuit 'circuit-54': T=542, qubits=55
[15/25] Circuit 'circuit-55': T=583, qubits=59
[16/25] Circuit 'circuit-56': T=625, qubits=63
[17/25] Circuit 'circuit-57': T=667, qubits=67
[18/25] Circuit 'circuit-58': T=708, qubits=71
[19/25] Circuit 'circuit-59': T=750, qubits=75
[20/25] Circuit 'circuit-60': T=791, qubits=79
[21/25] Circuit 'circuit-61': T=833, qubits=83
[22/25] Circuit 'circuit-62'

*39 metrics total — **35 shared** | **4 framework-specific*.*

,Metric,"Azure QDK (err_rate=1e-03, budget=0.01)","Qualtran (d=3, p=1e-03)",Ratio (B/A)
0,Logical qubits,2,2,1.000×
1,Logical depth,3,3,1.000×
2,Logical cycles,1,6.333,6.333×
3,T count,1,1,1.000×
4,T depth,1,1,1.000×
5,T count (from circuit),1,1,1.000×
6,Clifford count,2,2,1.000×
7,Rotation count,0,0,—
8,Toffoli count,0,0,—
9,Measurement count,0,0,—



Metrics that differ:


,Metric,"Azure QDK (err_rate=1e-03, budget=0.01)","Qualtran (d=3, p=1e-03)",Ratio (B/A)
0,Logical cycles,1,6.333,6.333×
1,Physical qubits (total),308,444,1.442×
2,Physical compute qubits,153,162,1.059×
3,Physical factory qubits,155,282,1.819×
4,Runtime (s),1.2e-06,7.6e-06,6.333×
5,Space-Time (qubit s),0.0003696,0.003374,9.130×
6,Logical error rate,0.0098,0.06853,6.993×
7,Physical qubits per logical qubit,154,222,1.442×
8,Factory qubit fraction,0.5032,0.6351,1.262×
9,Runtime per T gate (s),1.2e-06,7.6e-06,6.333×


WHY DO THE ESTIMATORS PRODUCE DIFFERENT RESULTS?

Both estimators receive the SAME canonical Clifford+T circuit, so any differences arise purely from the resource estimation models, not the input circuit.

ROTATION CONTEXT:
  • Azure QDK (err_rate=1e-03, budget=0.01): genuinely no Rz rotations — the input circuit has zero arbitrary-angle Rz gates. t_per_rotation is meaningless (reported as N/A).
  • Qualtran (d=3, p=1e-03): genuinely no Rz rotations — the input circuit has zero arbitrary-angle Rz gates. t_per_rotation is meaningless (reported as N/A).

Key model differences:

1. T-gate synthesis for arbitrary Rz rotations
   • Azure QDK: reports NUM_TS_PER_ROTATION (the actual synthesis count
     it uses internally for the given error budget).
   • Qualtran: counts raw Rz gates from the bloq graph; T synthesis cost
     is estimated post-hoc via the Solovay-Kitaev formula ~3·log₂(1/ε).
   → Expect T-count differences when rotation_count > 0. When rotation_count == 0
     both estimato

### Results table

Each row is one (circuit, estimator) pair.  Empty cells indicate unavailable metrics.


In [6]:
benchmark_df

,circuit_name,estimator_family,t_count,clifford_count,rotation_count,toffoli_count,measurement_count,runtime_seconds,total_qubits,compute_qubits,factory_qubits,space_time_volume,code_distance,logical_error_rate,error_budget,physical_error_rate,logical_qubits,logical_cycles,factory_count,num_factories
0,circuit-41,Azure,1,2,0,0,0,1.2e-06,308,153,155,0.0003696,3,0.0098,0.01,0.001,2,1,1×T,1
1,circuit-41,Qualtran,1,2,0,0,0,7.6e-06,444,162,282,0.003374,3,0.06853,0.01,0.001,2,6.333,15to1×1 (282 qubits),1
2,circuit-42,Azure,42,84,0,0,0,0.0001512,16940,3220,13720,2.561,9,0.009199,0.01,0.001,6,42,7×T,7
3,circuit-42,Qualtran,42,84,0,0,0,0.0001512,20502,3240,17262,3.1,9,0.004469,0.01,0.001,6,42,"15to1×7 (2,466 each)",7
4,circuit-43,Azure,84,168,0,0,0,0.0002352,49470,2910,46560,11.64,7,0.009352,0.01,0.001,10,84,12×T,12
5,circuit-43,Qualtran,84,168,0,0,0,0.0002352,20916,2940,17976,4.919,7,0.08715,0.01,0.001,10,84,"15to1×12 (1,498 each)",12
6,circuit-44,Azure,125,250,0,0,0,0.00045,41360,6440,34920,18.61,9,0.004167,0.01,0.001,14,125,9×T,9
7,circuit-44,Qualtran,125,250,0,0,0,0.00045,28674,6480,22194,12.9,9,0.02005,0.01,0.001,14,125,"15to1×9 (2,466 each)",9
8,circuit-45,Azure,167,334,0,0,0,0.0006012,46689,7889,38800,28.07,9,0.006018,0.01,0.001,18,167,10×T,10
9,circuit-45,Qualtran,167,334,0,0,0,0.0006012,32598,7938,24660,19.6,9,0.03085,0.01,0.001,18,167,"15to1×10 (2,466 each)",10


## Comparison Plots

Each plot compares **one Azure estimate** against **one Qualtran estimate**
for every circuit.  The x-axis is T count (sorted ascending).  Both estimators
are plotted as distinct curves so differences in scaling and magnitude are
immediately visible.

The DataFrame ``benchmark_df`` is the source for all plots — it retains every
metric returned by each estimator; the plots simply select the columns they need.


In [7]:
def _ensure_aligned_numeric(*cols):
    """Convert multiple columns to numeric and align their NaN positions.

    Unlike _try_numeric which drops independently per column, this function
    finds rows where *all* given columns have valid numeric values (no NaN,
    no inf, no string) — guaranteeing identical index sets across all returned
    Series.

    Parameters
    ----------
    *cols : pd.Series
        Any number of columns to align simultaneously.

    Returns
    -------
    list[pd.Series]
        One numeric Series per input column, all sharing the same index.
    """
    import numpy as np
    # Find rows where every column is a finite number
    valid = pd.DataFrame({i: c for i, c in enumerate(cols)})
    num_valid = valid.apply(pd.to_numeric, errors='coerce')
    num_valid.replace([np.inf, -np.inf], np.nan, inplace=True)
    mask = num_valid.notna().all(axis=1)

    # Debug: report which rows were dropped
    if not mask.all():
        n_dropped = (~mask).sum()
        for col_name in valid.columns:
            col_nans = num_valid[col_name].isna().sum()
            if col_nans > 0:
                print(f"  ⚠️  {col_name}: {col_nans} non-numeric/NaN values dropped")
        print(f"  ℹ️  Row alignment: {mask.sum()}/{mask.sum()+n_dropped} rows have all valid numeric values")

    aligned = [num_valid.iloc[:, i][mask] for i in range(num_valid.shape[1])]
    return aligned


# ---------------------------------------------------------------------------
# Plot 1 -- T Count vs Space-Time
# Y-axis = total physical qubits * runtime (qubit-seconds)
# Each estimator produces ONE continuous line.
# ---------------------------------------------------------------------------

def plot_space_time(df, filename=None):
    # --- Diagnostic output ---
    # print(f"[Space-Time] DataFrame shape: {df.shape}")
    # print(f"  t_count dtype: {df['t_count'].dtype}, space_time_volume dtype: {df['space_time_volume'].dtype}")
    # t_nan = df['t_count'].isna().sum() if pd.api.types.is_numeric_dtype(df['t_count']) else df['t_count'].apply(lambda x: str(x) in ('nan', 'NaN', '')).sum()
    # st_nan = df['space_time_volume'].isna().sum()
    # print(f"  t_count NaN/non-numeric: {t_nan}, space_time_volume NaN: {st_nan}")

    fig, ax = plt.subplots(figsize=(9, 6))
    fig.set_dpi(150)

    valid = df.dropna(subset=['t_count', 'space_time_volume']).copy()
    if valid.empty:
        # If dropna left nothing, try _ensure_aligned_numeric as fallback
        t_series, st_series = _ensure_aligned_numeric(df['t_count'], df['space_time_volume'])
        if t_series is None or len(t_series) == 0:
            print("Warning: No valid data for Space-Time plot.")
            return None
        valid = pd.DataFrame({'t_count': t_series, 'space_time_volume': st_series})

    # Group by estimator_family (simplified name) so all points from the same
    # estimator family form one continuous line, regardless of config details.
    for name in sorted(valid['estimator_family'].unique()):
        grp = valid[valid['estimator_family'] == name].sort_values('t_count')
        ax.plot(grp['t_count'], grp['space_time_volume'],
                '-o', label=name, markersize=3, linewidth=1.5)

    ax.set_xlabel('T count')
    ax.set_ylabel('Space-Time (qubit$\\cdot$s)')
    ax.set_title('Space-Time vs T Count')
    ax.legend()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3)

    if filename:
        fig.savefig(filename, bbox_inches='tight', dpi=150)
        plt.close(fig)
    else:
        plt.show()
    return fig


# ---------------------------------------------------------------------------
# Plot 2 -- T Count vs Physical Qubits (stacked area via fill_between)
# For each estimator: compute (color 1), factory (color 2), total (color 3).
# Three distinct colors -- total is a line drawn over the filled regions.
# ---------------------------------------------------------------------------

def plot_qubit_breakdown(df, filename=None):
    estimators = sorted(df["estimator_family"].unique())

    if not estimators:
        print("Warning: No estimator data.")
        return None

    ESTIMATOR_COLORS = {
        "Azure": {
            "compute": "#9ecae1",
            "factory": "#3182bd",
            "total": "#08519c",
        },
        "Qualtran": {
            "compute": "#fdd0a2",
            "factory": "#f16913",
            "total": "#a63603",
        },
    }

    fig, axes = plt.subplots(
        1,
        len(estimators),
        figsize=(6 * len(estimators), 6),
        dpi=150,
    )

    # If only one estimator exists, axes isn't iterable.
    if len(estimators) == 1:
        axes = [axes]


    for ax, est_name in zip(axes, estimators):

        grp = df[df["estimator_family"] == est_name].copy()

        # Use row-wise alignment so all columns share the same index.
        t_raw, compute_raw, factory_raw = _ensure_aligned_numeric(
            grp['t_count'],
            grp['compute_qubits'],
            grp['factory_qubits'],
        )

        if len(t_raw) == 0:
            print(f"  [{est_name}] No rows with all three numeric columns -- skipping.")
            # Still draw an empty axes so sharey stays consistent
            ax.set_title(est_name)
            ax.set_xlabel("T count")
            ax.set_ylabel("Physical Qubits")
            ax.text(0.5, 0.5, 'No valid data', ha='center', va='center', transform=ax.transAxes)
            continue

        valid_any = True
        # Sort by t and extract values (NOT .loc — the three Series have different indices
        # because _ensure_aligned_numeric dropped different rows per column).
        paired = pd.DataFrame({
            't': t_raw.values,
            'compute': compute_raw.values,
            'factory': factory_raw.values,
        })
        paired.sort_values('t', inplace=True)
        paired.reset_index(drop=True, inplace=True)
        t = paired['t']
        compute = paired['compute']
        factory = paired['factory']
        total = compute + factory

        if "azure" in est_name.lower():
            colors = ESTIMATOR_COLORS["Azure"]
        else:
            colors = ESTIMATOR_COLORS["Qualtran"]

        # --- Diagnostic output ---
        # print(f"  [{est_name}] {len(t)} aligned points, qubit range: [compute={compute.min():.0f} -- factory+compute={total.max():.0f}]")

        ax.fill_between(
            t,
            0,
            compute,
            color=colors["compute"],
            alpha=0.35,
            label="Compute",
        )

        ax.fill_between(
            t,
            compute,
            total,
            color=colors["factory"],
            alpha=0.55,
            label="Factory",
        )

        ax.plot(
            t,
            total,
            color=colors["total"],
            linewidth=2.5,
            marker="o",
            markersize=3,
            label="Total",
        )

        ax.set_title(est_name)
        ax.set_xlabel("T count")
        ax.grid(True, alpha=0.3)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.legend()

    axes[0].set_ylabel("Physical Qubits")

    fig.suptitle("Physical Qubits vs T Count")

    # Only apply tight_layout if at least one subplot has data.
    # Empty subplots (all `continue`d above) cause _get_tick_label_size -> NaN
    # which crashes the layout pass as int(np.floor(length / size)).
    if valid_any:
        plt.tight_layout()

    if filename:
        fig.savefig(filename, bbox_inches="tight", dpi=150)
        plt.close(fig)
    else:
        # print("Figure size:", fig.get_size_inches())
        # print("Figure dpi:", fig.dpi)

        # for i, ax in enumerate(axes):
        #     print(f"\nAxis {i}")
        #     print("position:", ax.get_position())
        #     print("xlim:", ax.get_xlim())
        #     print("ylim:", ax.get_ylim())
        plt.show()

    return fig


# ---------------------------------------------------------------------------
# Plot 3 -- T Count vs Runtime
# Each estimator produces ONE continuous line (log scale on y-axis).
# ---------------------------------------------------------------------------

def plot_runtime(df, filename=None):
    # # --- Diagnostic output ---
    # print(f"[Runtime] DataFrame shape: {df.shape}")
    # t_nan = df['t_count'].isna().sum() if pd.api.types.is_numeric_dtype(df['t_count']) else df['t_count'].apply(lambda x: str(x) in ('nan', 'NaN', '')).sum()
    # rt_nan = df['runtime_seconds'].isna().sum()
    # print(f"  t_count NaN: {t_nan}, runtime_seconds NaN: {rt_nan}")

    fig, ax = plt.subplots(figsize=(9, 6))
    fig.set_dpi(150)

    valid = df.dropna(subset=['t_count', 'runtime_seconds']).copy()
    if valid.empty:
        # Fallback: use _ensure_aligned_numeric
        t_series, rt_series = _ensure_aligned_numeric(df['t_count'], df['runtime_seconds'])
        if t_series is None or len(t_series) == 0:
            print("Warning: No valid data for Runtime plot.")
            return None
        valid = pd.DataFrame({'t_count': t_series, 'runtime_seconds': rt_series})

    for name in sorted(valid['estimator_family'].unique()):
        grp = valid[valid['estimator_family'] == name].sort_values('t_count')
        ax.plot(grp['t_count'], grp['runtime_seconds'],
                '-o', label=name, markersize=3, linewidth=1.5)

    ax.set_xlabel('T count')
    ax.set_ylabel('Runtime (seconds)')
    ax.set_title('Runtime vs T Count')
    ax.legend()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')

    if filename:
        fig.savefig(filename, bbox_inches='tight', dpi=150)
        plt.close(fig)
    else:
        plt.show()
    return fig


print("Plotting helpers loaded.")


Plotting helpers loaded.


In [8]:
# import numpy as np

# # Debug output before plotting
# print("\n=== Plotting Diagnostic ===")
# print(f"benchmark_df shape: {benchmark_df.shape}")
# print("Columns used for physical qubit plots:", ['t_count', 'compute_qubits', 'factory_qubits', 'total_qubits'])
# nan_counts = benchmark_df[['t_count', 'compute_qubits', 'factory_qubits', 'total_qubits']].isna().sum()
# print(f"NaN counts in key columns:\n{nan_counts}")
# # Check for inf values in float columns
# float_cols = benchmark_df.select_dtypes(include=[np.floating]).columns
# inf_counts = benchmark_df[float_cols].replace([np.inf, -np.inf], np.nan).isna().sum()
# print(f"Inf counts in float columns:\n{inf_counts[inf_counts > 0]}")
# # Circuits with NaN/inf
# key_plot_cols = ['t_count', 'compute_qubits', 'factory_qubits', 'total_qubits']
# nan_rows = benchmark_df[benchmark_df[key_plot_cols].isna().any(axis=1)]
# if not nan_rows.empty:
#     print(f"\nCircuits with NaN values in key plot columns ({len(nan_rows)} rows):")
#     for _, row in nan_rows.iterrows():
#         print(f"  {row['circuit_name']:20s} {row['estimator_family']:12s}")
# print(f"\nCircuits per estimator:")
# for est in benchmark_df['estimator_family'].unique():
#     est_rows = benchmark_df[benchmark_df['estimator_family'] == est]
#     print(f"  {est}: {len(est_rows)} rows")

from pathlib import Path

output_dir = Path("figs") / f"{c_count}_{t_count}_{q_count}"
output_dir.mkdir(parents=True, exist_ok=True)  # Create the folder if it doesn't exist

# Plot 1 -- Space-Time
output_path = output_dir / "lin_spacetime"
fig1 = plot_space_time(benchmark_df, output_path)

# Plot 2 -- Physical Qubit Breakdown (the one that was crashing)
output_path = output_dir / "lin_qubit"
fig2 = plot_qubit_breakdown(benchmark_df, output_path)

# Plot 3 -- Runtime
output_path = output_dir / "lin_runtime"
fig3 = plot_runtime(benchmark_df, output_path)


In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def plot_estimator_ratios(df, filename=None):
    """
    Plot Qualtran/Azure ratios versus T count.

    Produces three vertically stacked plots:

        Physical Qubits Ratio
        Runtime Ratio
        Space-Time Ratio

    Ratio = Qualtran / Azure

    Requires columns

        estimator_family
        t_count
        physical_qubits
        runtime_seconds
        space_time_volume
    """

    # ------------------------------------------------------------
    # Split Azure and Qualtran
    # ------------------------------------------------------------

    azure = (
        df[df["estimator_family"].str.contains("Azure", case=False)]
        .copy()
        .sort_values("t_count")
    )

    qualtran = (
        df[df["estimator_family"].str.contains("Qualtran", case=False)]
        .copy()
        .sort_values("t_count")
    )

    if azure.empty or qualtran.empty:
        print("Need both Azure and Qualtran results.")
        return

    # ------------------------------------------------------------
    # Merge on T count
    # ------------------------------------------------------------

    merged = pd.merge(
        azure,
        qualtran,
        on="t_count",
        suffixes=("_azure", "_qualtran"),
    )

    if merged.empty:
        print("No matching T counts between Azure and Qualtran.")
        return

    # ------------------------------------------------------------
    # Compute ratios
    # ------------------------------------------------------------

    merged["qubit_ratio"] = (
        merged["total_qubits_qualtran"]
        / merged["total_qubits_azure"]
    )

    merged["runtime_ratio"] = (
        merged["runtime_seconds_qualtran"]
        / merged["runtime_seconds_azure"]
    )

    merged["space_time_ratio"] = (
        merged["space_time_volume_qualtran"]
        / merged["space_time_volume_azure"]
    )

    # ------------------------------------------------------------
    # Plot
    # ------------------------------------------------------------

    fig, axes = plt.subplots(
        3,
        1,
        figsize=(9, 10),
        dpi=150,
        sharex=True,
    )

    ratios = [
        ("qubit_ratio", "Physical Qubit Ratio"),
        ("runtime_ratio", "Runtime Ratio"),
        ("space_time_ratio", "Space-Time Ratio"),
    ]

    for ax, (col, title) in zip(axes, ratios):

        ax.plot(
            merged["t_count"],
            merged[col],
            "-o",
            linewidth=2,
            markersize=3,
        )

        ax.axhline(
            1.0,
            color="black",
            linestyle="--",
            linewidth=1,
            alpha=0.7,
        )

        ax.set_ylabel("Qualtran / Azure")
        ax.set_title(title)

        ax.grid(alpha=0.3)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[-1].set_xlabel("T count")

    fig.suptitle(
        "Qualtran vs Azure Resource Ratios",
        fontsize=14,
    )

    plt.tight_layout()

    if filename:
        plt.savefig(filename, dpi=150, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()

    return fig

In [10]:
from pathlib import Path

output_dir = Path("figs") / f"{c_count}_{t_count}_{q_count}"
output_dir.mkdir(parents=True, exist_ok=True) 

output_path = output_dir / "lin_ratio"

fig4 = plot_estimator_ratios(benchmark_df, output_path)

## Side-by-Side Comparison

Displaying estimator results for the first circuit as a side-by-side table.


In [11]:
est_rows = benchmark_df[benchmark_df['estimator_family'] != 'Transpiler']
circuit_names = est_rows['circuit_name'].unique()

if len(circuit_names) == 0:
    print("No estimator results to compare.")
else:
    first_circuit = circuit_names[0]
    subset = est_rows[est_rows['circuit_name'] == first_circuit]
    est_names_subset = subset['estimator_family'].unique()

    if len(est_names_subset) < 2:
        print("Need >=2 estimators for comparison.")
    else:
        # Pivot to side-by-side format
        pivot_cols = [c for c in subset.columns
                      if c not in ('circuit_name', 'estimator_family')]
        pivot_df = subset.pivot(index='circuit_name', columns='estimator_family', values=pivot_cols)
        display(pivot_df)

        # Summary stats
        print(f"\nCircuit     : {first_circuit}")
        for name in est_names_subset:
            row = subset[subset['estimator_family'] == name].iloc[0]
            print(f"\nEstimator   : {name}")
            print(f"  T count       : {row.get('t_count', 'N/A')}")
            print(f"  Runtime (s)   : {row.get('runtime_seconds', 'N/A')}")
            tq = row.get('total_qubits')
            cq = row.get('compute_qubits')
            fq = row.get('factory_qubits')
            st = row.get('space_time_volume')
            print(f"  Total qubits  : {tq:,}" if pd.notna(tq) else f"  Total qubits  : {tq}")
            print(f"  Compute qubits: {cq:,}" if pd.notna(cq) else f"  Compute qubits: {cq}")
            print(f"  Factory qubits: {fq:,}" if pd.notna(fq) else f"  Factory qubits: {fq}")
            print(f"  Space-Time    : {st}" if pd.notna(st) else f"  Space-Time    : N/A")


t_count          clifford_count          rotation_count  \
estimator_family   Azure Qualtran          Azure Qualtran          Azure   
circuit_name                                                               
circuit-41             1        1              2        2              0   

                          toffoli_count          measurement_count           \
estimator_family Qualtran         Azure Qualtran             Azure Qualtran   
circuit_name                                                                  
circuit-41              0             0        0                 0        0   

                  ... physical_error_rate          logical_qubits           \
estimator_family  ...               Azure Qualtran          Azure Qualtran   
circuit_name      ...                                                        
circuit-41        ...               0.001    0.001              2        2   

                 logical_cycles          factory_count                        \
estimator_family          Azure Qualtran         Azure              Qualtran   
circuit_name                                                                   
circuit-41                    1    6.333           1×T  15to1×1 (282 qubits)   

                 num_factories           
estimator_family         Azure Qualtran  
circuit_name                             
circuit-41                   1        1  

[1 rows x 36 columns]


Circuit     : circuit-41

Estimator   : Azure
  T count       : 1
  Runtime (s)   : 1.2e-06
  Total qubits  : 308
  Compute qubits: 153
  Factory qubits: 155
  Space-Time    : 0.0003696

Estimator   : Qualtran
  T count       : 1
  Runtime (s)   : 7.6e-06
  Total qubits  : 444
  Compute qubits: 162
  Factory qubits: 282
  Space-Time    : 0.0033744


In [12]:
from pathlib import Path

output_dir = Path("results")
output_dir.mkdir(exist_ok=True)  # Create the folder if it doesn't exist

output_path = output_dir / f"{c_count}_{t_count}_{q_count}_lin_results.csv"
benchmark_df.to_csv(output_path, index=False)

print(f"Saved {len(benchmark_df)} rows to {output_path}")

Saved 50 rows to results/25_1000_100_lin_results.csv


## Appendix - Custom Circuit List

To benchmark **your own circuits** instead of the generated ones, replace `circuit_list`
with any Python list of `QuantumCircuit` objects or file paths:

```python
from qiskit import QuantumCircuit

# Hand-built circuits
qc_a = QuantumCircuit(3, 0)
qc_a.h(0)
qc_a.cx(0, 1)
circuit_list = [qc_a]

# Or load from OpenQASM files
circuit_list = ["path/to/circuit1.qasm", "path/to/circuit2.qasm"]

# Then re-run:
benchmark_df = run_benchmark(circuit_list, cfg)
```

The notebook handles both formats transparently.
